In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_dataset, load_fine


DATA_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

fine = load_fine(
    DATA_DIR,
    "fine.csv",
)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Aggregate fine-level regions into mid-level regions using the predefined
# anatomical mapping

from utils.io import aggregate_regions

mid = (
    fine.groupby(level="mid", sort=False)
    .mean()
    .round(2)
)

aggregation_dictionary = pd.read_csv(
    DATA_DIR / "dictionary_for_aggregated.csv",
    sep=";",
)

aggregated_mid = aggregate_regions(
    mid,
    aggregation_dictionary,
).round(2)

"""
aggregated_mid.to_csv(
    DATA_DIR / "aggregated_mid.csv",sep=";"
)
"""

aggregated_mid

In [ ]:
# Normalize each region to the mean HC value and plot the resulting heatmap

from utils.plot import plot_grouped_heatmap

hc_columns = [col for col in aggregated_mid.columns if col.startswith("HC")]

hc_mean = aggregated_mid[hc_columns].mean(axis=1)
normalized = aggregated_mid.div(hc_mean, axis=0).T

group_sizes = {
    "HC": 5,
    "CNTX": 7,
    "OCT": 6,
    "OPCRT": 7,
}

fig, ax = plot_grouped_heatmap(
    normalized,
    group_sizes,
    colorbar_label="c-Fos$^+$/mm$^2$ relative to HC",
)

fig.savefig(
     OUTPUT_DIR / "aggregated_heatmap_horizontal.svg",
     bbox_inches="tight",
)

In [ ]:
# Perform PLS analysis to identify brain regions contributing to differences
# between the experimental groups

from utils.statistics import prepare_pls_dataframe, perform_pls_df

pls_dataset = aggregated_mid

pls_hc = prepare_pls_dataframe(
    pls_dataset,
    ["HC", "OPCRT"],
)

pls_cntx = prepare_pls_dataframe(
    pls_dataset,
    ["CNTX", "OPCRT"],
)

pls_oct = prepare_pls_dataframe(
    pls_dataset,
    ["OCT", "OPCRT"],
)

bootstrap_threshold = 2.58  

vpd_hc, significant_regions_hc = perform_pls_df(
    pls_hc,
    n_bootstrap=10000,
    threshold=bootstrap_threshold,
)

vpd_cntx, significant_regions_cntx = perform_pls_df(
    pls_cntx,
    n_bootstrap=10000,
    threshold=bootstrap_threshold,
)

vpd_oct, significant_regions_oct = perform_pls_df(
    pls_oct,
    n_bootstrap=10000,
    threshold=bootstrap_threshold,
)

In [ ]:
# Save the PLS analysis results and significant regions

pls_hc.to_csv(
    DATA_DIR / "pls_dataset_HC_OPCRT.csv",
    index=False,
)

pls_cntx.to_csv(
    DATA_DIR / "pls_dataset_CNTX_OPCRT.csv",
    index=False,
)

pls_oct.to_csv(
    DATA_DIR / "pls_dataset_OCT_OPCRT.csv",
    index=False,
)

vpd_hc.to_csv(
    DATA_DIR / "pls_bootstrap_ratio_HC_OPCRT.csv",
)

vpd_cntx.to_csv(
    DATA_DIR / "pls_bootstrap_ratio_CNTX_OPCRT.csv",
)

vpd_oct.to_csv(
    DATA_DIR / "pls_bootstrap_ratio_OCT_OPCRT.csv",
)


significant_regions = pd.concat(
    {
        "HC_OPCRT": significant_regions_hc,
        "CNTX_OPCRT": significant_regions_cntx,
        "OCT_OPCRT": significant_regions_oct,
    },
    axis=1,
)

significant_regions.to_csv(
    DATA_DIR / "pls_significant_regions.csv",
)

In [ ]:
# Visualize the PLS results for each group comparison

from utils.plot import get_colors_for_regions, plot_vpd, plot_sig_regions

colors_aggregated = pd.read_csv(
    DATA_DIR / "aggregated_colors.csv",
    sep=";",
    index_col=0
).drop_duplicates(['target']).drop('source',axis=1)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 20,
})

pls_results = {
    "HC_vs_OPCRT": vpd_hc,
    "CNTX_vs_OPCRT": vpd_cntx,
    "OCT_vs_OPCRT": vpd_oct,
}

for comparison, vpd_result in pls_results.items():

    fig, ax = plt.subplots(
        figsize=(24, 6),
    )

    plot_vpd(
        ax,
        vpd_result,
        title=comparison.replace("_", " "),
        colors_df=colors_aggregated,
        width=0.8,
    )

    plt.tight_layout()

    fig.savefig(
        OUTPUT_DIR / f"PLS_{comparison}.svg",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)